# 01 - Data profiling

Stage 1: what is actually in this dataset, and what is not.

Run `python run_pipeline.py` for the full audit. This notebook is for exploring interactively.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

from src import config, preprocessing, data_profiling, feature_engineering
from src.utils import setup_logging
setup_logging()

# Use --nrows style subsetting here to keep the notebook responsive.
# PaySim is time-ordered, so this is an EARLY TIME WINDOW, not a random sample.
df = preprocessing.load_paysim(nrows=1_000_000)
df = preprocessing.prepare(df)
df.head()

## Class balance and why accuracy misleads


In [ ]:
balance = data_profiling.profile_target(df)
balance

A model predicting 'legitimate' every time achieves the accuracy shown above and catches zero fraud. That is why this project reports PR-AUC, capture rate and false-positive rate instead.


## Account repetition: the check that decides the Stage 2 design


In [ ]:
accounts = data_profiling.profile_accounts(df)
print(accounts['design_decision'])
accounts['prior_history_coverage']

## Leakage audit

The dataset documentation states fraudulent transactions are cancelled, so balance fields describe the aftermath. Here is the measured separation.


In [ ]:
audit = data_profiling.leakage_audit(df)
import pandas as pd
pd.DataFrame(audit['decision_table'])

In [ ]:
pd.DataFrame(audit['balance_pattern_checks_pct']).T

## Honest baselines

No model exists yet, so precision and recall are computed for reference rules, not for an imaginary classifier.


In [ ]:
baselines = data_profiling.honest_baselines(df)
pd.DataFrame(baselines['rules']).T

In [ ]:
print(baselines['why_accuracy_misleads'])
print()
print(baselines['why_pr_auc_matters'])